# Run the Agent

The travel agent is fully wired — LLM Gateway (Module 2) for models, AgentCore Gateway
(Module 3) for tools. Now test it end-to-end.

The primary UX is the React frontend on Amplify. This notebook also shows how to call the
agent programmatically using the AgentCore Runtime API (same auth flow, no browser required).


## Step 1: Print the Amplify URL

Log in with `workshop@example.com` and the temporary password printed by notebook 02 and set a new password on
first login.


In [ ]:
import os
import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )
cfn = boto3.client("cloudformation", region_name=REGION)

outputs = {
    o["OutputKey"]: o["OutputValue"]
    for o in cfn.describe_stacks(StackName="FAST-stack")["Stacks"][0]["Outputs"]
}
AMPLIFY_URL = outputs["AmplifyUrl"]
RUNTIME_ARN = outputs["RuntimeArn"]
COGNITO_USER_POOL_ID = outputs["CognitoUserPoolId"]

print(f"Amplify URL:    {AMPLIFY_URL}")
print(f"Runtime ARN:    {RUNTIME_ARN}")

## Step 2: Test 1 — Plan a Trip (via UI)

Send this message in the Amplify chat:

> Plan a trip from SFO to Tokyo for 2026-09-15 to 2026-09-18, 2 guests, budget $2000 for flights

Expected behaviour:

1. Agent calls `search_flights_by_budget` (SFO → TYO, < $2000)
2. Agent calls `search_hotels` for Tokyo
3. Agent uses the Code Interpreter to total costs
4. Agent replies with a structured itinerary

Tool invocations shown in chat will look like `gw_tg-workshop-flights-mcp___search_flights_by_budget` —
confirming that they flow through the AgentCore Gateway.


## Step 3: Test 2 — Multi-Turn Memory (via UI)

Follow up with:

> Can you find me a cheaper hotel option? Under $100 per night.

The agent should remember the Tokyo trip context and call `search_hotels_by_budget` with the
new constraint. That proves AgentCore Memory is persisting conversation state across turns.


## Step 4: Test 3 — Code Interpreter (via UI)

> Compare the total cost of the ANA flight + Sakura Inn vs the JAL flight + Tokyo Bay Resort
> for 2 guests, 3 nights. Show me a breakdown.

The agent should use the Code Interpreter to produce the comparison table.


## Step 5: (Advanced) Programmatic Invocation

The runtime is fronted by a Cognito JWT authorizer (configured in the FAST CDK — `RuntimeAuthorizerConfiguration.usingJWT(...)`). That means `boto3.invoke_agent_runtime()` **will not work directly** — boto3 signs with SigV4, the runtime rejects SigV4, and you will get `401`/`403`.

To call the runtime without a browser you need to:

1. Authenticate the workshop user against Cognito (`InitiateAuth` with `USER_PASSWORD_AUTH` using the **permanent** password you chose on first UI login — not the temporary `Workshop1!`).
2. Take the returned `IdToken` and `POST` it as a `Bearer` header to `https://{RUNTIME_ID}.runtime.bedrock-agentcore.{REGION}.amazonaws.com/invocations?qualifier=DEFAULT`.

For this workshop the Amplify UI already does all of that for you, so we skip the programmatic smoke test. The UI exercise above is sufficient proof the runtime, tools, memory, and observability are all wired correctly.


## What Just Happened

Every interaction exercised the full platform:

| Layer | What Happened |
|-------|--------------|
| Frontend | React app on Amplify sent the prompt |
| User auth | Cognito JWT validated by the runtime |
| Model call | LLM Gateway (Module 2) with virtual-key budget tracking |
| Tool discovery | Agent called `tools/list` on Module 3's gateway over MCP |
| Tool invocation | Gateway dispatched to Flights/Hotels Lambda targets |
| M2M auth | Token Vault fetched a JWT from Module 3's Cognito |
| Memory | AgentCore Memory persisted conversation turns |
| Code interpreter | Python ran in AgentCore's secure sandbox |
| Observability | OpenTelemetry traces exported to CloudWatch |

Next: inspect those traces in **notebook 06 — Observe**.
